In [0]:
dbutils.secrets.listScopes()

In [0]:
# Setting up the secret scopes 
spnclientID = dbutils.secrets.get(scope = 'spnclientid', key = 'formula1SPNid')
spnclientSecret = dbutils.secrets.get(scope = 'spnclientsecret', key = 'formula1SPNsecret')
spnTenantID = dbutils.secrets.get(scope = 'spntenantid', key = 'formula1Tenantid')

In [0]:
# Authenticate
storage_account = {"storage account name"}
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    spnclientID
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    spnclientSecret
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{spnTenantID}/oauth2/token"
)
# Validate
container = {"container name"}

base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/ecommercedata/landing"

# Confirming the csv files are present
files = dbutils.fs.ls(base_path)
if len(files) > 0:
    print(f"Found {len(files)} files in {base_path}")
    display(files)
else:
     print(f"No files found in {base_path}")

In [0]:
# Reading csv files 

exception_name_mapping = {
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_order_items_dataset.csv": "order_items"
}
dataframes = {}

for file in files:
    if file.name.endswith(".csv"):
        
        # ✅ Check exception first
        if file.name in exception_name_mapping:
            second_name = exception_name_mapping[file.name]
        
        else:
            # Default behaviour: split by "_"
            filename = file.name.replace(".csv", "")
            parts = filename.split("_")
            second_name = parts[1] if len(parts) >= 2 else "unknown"

        df_name = f"df_{second_name}"
        print(f"Loading {file.name} as {df_name}")

        df = (
            spark.read.format("csv")\
            .option("header", "true")\
            .option("inferSchema", "true")\
            .load(file.path)
        )

        dataframes[df_name] = df
# Creating all csv as different dataframes

for df_name, df in dataframes.items():
    globals()[df_name] = df          # Python usage
    df.createOrReplaceTempView(df_name)  # SQL usage

In [0]:
%sql 
select * from df_customers

In [0]:
%sql
-- Joining customer, product and order table to see reorder status, reorder state, products most reordered
WITH customer_product_orders AS (
    SELECT
        c.customer_unique_id,
        o.order_id,
        oi.product_id,
        c.customer_city,
        c.customer_state,
        YEAR(o.order_purchase_timestamp)  AS order_year,
        MONTH(o.order_purchase_timestamp) AS order_month
    FROM df_customers c
    JOIN df_orders o
        ON c.customer_id = o.customer_id
    JOIN df_order_items oi
        ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
)
SELECT
    customer_unique_id,
    product_id,
    customer_city,
    customer_state,
    order_year,
    order_month,
    COUNT(DISTINCT order_id) AS order_count,
    COLLECT_SET(order_id) AS order_ids
FROM customer_product_orders
GROUP BY
    customer_unique_id,
    product_id,
    customer_city,
    customer_state,
    order_year,
    order_month
HAVING COUNT(DISTINCT order_id) > 1
ORDER BY order_year DESC, order_month, order_count DESC;


In [0]:
%sql
-- Understanding the order_id in payment_records table to see if multiple payment records per order_id
SELECT
    order_id,
    COUNT(*) AS row_count
FROM df_order_payments
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC;

In [0]:
%sql

-- Checking for any duplicate rows in payment records

SELECT
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value,
    COUNT(*) AS duplicate_count
FROM df_order_payments
GROUP BY
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Joining payments table and customer prodcut order table 

WITH customer_payments_agg AS (
    SELECT
        order_id,
        SUM(payment_value) AS total_payment_value
    FROM df_order_payments
    GROUP BY order_id
),
customer_product_orders AS (
    SELECT
        c.customer_unique_id,
        o.order_id,
        oi.product_id,
        c.customer_city,
        c.customer_state,
        YEAR(o.order_purchase_timestamp)  AS year,
        MONTH(o.order_purchase_timestamp) AS month,
        p.total_payment_value
    FROM df_customers c
    JOIN df_orders o
        ON c.customer_id = o.customer_id
    JOIN df_order_items oi
        ON o.order_id = oi.order_id
    LEFT JOIN customer_payments_agg p       -- ✅ correct join
        ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
)
SELECT
    customer_unique_id,
    product_id,
    customer_city,
    customer_state,
    year,
    month,
    COUNT(DISTINCT order_id) AS order_count,
    COLLECT_SET(order_id)    AS order_ids,
    SUM(total_payment_value) AS total_payment_value
FROM customer_product_orders
GROUP BY
    customer_unique_id,
    product_id,
    customer_city,
    customer_state,
    year,
    month
HAVING COUNT(DISTINCT order_id) > 1
ORDER BY year, month, order_count DESC;

-- This table can give us 
-- -Customer who reordered the most \
-- -Produt most reorderd in a specifc city 
-- -which product gave us the maximum revenue
-- -which city gave us the maximum revenue
-- -which year we had the more revenue 
-- -which moth we had the more revenue


In [0]:
# Top seller with maximum revenue
df_top_seller_revenue = spark.sql("""
    SELECT
        seller_id,
        SUM(price) AS total_revenue
    FROM df_order_items
    GROUP BY seller_id
    ORDER BY total_revenue DESC
""")
df_top_seller_revenue.display()


In [0]:
# Top seller with maximum orders 
df_top_seller_orders = spark.sql("""
    SELECT
        seller_id,
        COUNT(DISTINCT order_id) AS total_orders
    FROM df_order_items
    GROUP BY seller_id
    ORDER BY total_orders DESC
    
""")
df_top_seller_orders.display()

In [0]:
df_customer_payments = spark.sql("""
    WITH customer_payments_agg AS (
        SELECT
            order_id,
            SUM(payment_value) AS total_payment_value
        FROM df_order_payments
        GROUP BY order_id
    ),
    customer_product_orders AS (
        SELECT
            c.customer_unique_id,
            o.order_id,
            oi.product_id,
            c.customer_city,
            c.customer_state,
            YEAR(o.order_purchase_timestamp)  AS year,
            MONTH(o.order_purchase_timestamp) AS month,
            p.total_payment_value
        FROM df_customers c
        JOIN df_orders o
            ON c.customer_id = o.customer_id
        JOIN df_order_items oi
            ON o.order_id = oi.order_id
        LEFT JOIN customer_payments_agg p
            ON o.order_id = p.order_id
        WHERE o.order_status = 'delivered'
    )
    SELECT
        customer_unique_id,
        product_id,
        customer_city,
        customer_state,
        year,
        month,
        COUNT(DISTINCT order_id) AS order_count,
        COLLECT_SET(order_id)    AS order_ids,
        SUM(total_payment_value) AS total_payment_value
    FROM customer_product_orders
    GROUP BY
        customer_unique_id,
        product_id,
        customer_city,
        customer_state,
        year,
        month
    HAVING COUNT(DISTINCT order_id) > 1
""")
df_customer_payments.display()

In [0]:
# Writing to silver layer 
output_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/ecommercedata/silver"
df_customer_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{output_path}/customerpayments")

df_top_seller_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{output_path}/sellerorders")

df_top_seller_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{output_path}/sellerrevenue")